## GroundTruth 만든 것으로 평가
- gpt-4o-model(1차), 정성적 평가(2차)

### 1. 정답용 dataset 가져오기

In [1]:
import pandas as pd

df = pd.read_csv("data/groundTruth/final_groundTruth_f1.csv")
df.describe()

,user_info,recipe_info,generation,groundTruth
count,35,35,35,35
unique,10,35,35,35
top,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '67610699846f9e5eb975e532', 'title': '...",{'main_changes_from_original_recipe': ['🥗 닭가슴살...,{'main_changes_from_original_recipe': ['🍗 연어 대...
freq,4,1,1,1


### 2. eval prompt로 test - toxicity
- 1,2,3번 모두 0.0

In [2]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

LangSmith 추적을 하지 않습니다.


In [3]:
import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


2024-12-23 05:24:12,730 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [3]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "toxicity"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 05:13:07,887 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 05:13:08,207 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:13:08,211 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 05:13:11,469 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:13:11,481 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:13:11,481 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:13:11,482 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 05:13:12,265 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:13:12,277 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:13:12,277 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:13:12,278 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 05:14:29,866 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:14:29,882 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:14:29,883 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:14:29,884 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 05:14:34,191 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [8]:
import pandas as pd
import ast

col_name = "toxicity"

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nlargest(3, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nReason이 '해당없음'가 아닌 행:")
print(non_x_reasons)
print("\nScore 상위 3개의 행:")
print(top3_scores)


Score 평균값: 0.005714285714285714

Reason이 '해당없음'가 아닌 행:
Empty DataFrame
Columns: [user_info, recipe_info, generation, groundTruth, toxicity, score, reason]
Index: []

Score 상위 3개의 행:
                                           user_info  \
3  {'user_allergy_ingredients': [], 'user_dislike...   
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   

                                         recipe_info  \
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   

                                          generation  \
3  {'original_recipe_food_group_composition': [{'...   
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   

                                         groundTruth  \
3  {'original_recipe_food_group_composition': [{'...   
0  {'original_recipe_food_group

In [8]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "toxicity"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 05:25:51,264 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:25:51,264 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:25:51,266 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 05:25:52,135 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:25:52,149 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:25:52,150 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:25:52,151 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 05:25:53,021 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:25:53,031 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:25:53,032 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:25:53,033 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 05:27:18,085 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:27:18,101 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:27:18,102 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:27:18,103 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 05:27:22,276 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [9]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "toxicity"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nlargest(3, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nReason이 '해당없음'가 아닌 행:")
print(non_x_reasons)
print("\nScore 상위 3개의 행:")
print(top3_scores)


Score 평균값: 0.0

Reason이 '해당없음'가 아닌 행:
Empty DataFrame
Columns: [user_info, recipe_info, generation, groundTruth, toxicity, score, reason]
Index: []

Score 상위 3개의 행:
                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   

                                         groundTruth  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{

### 2. eval prompt로 test - contextcorrectness
- 1,2,3번 모두 0.0

In [1]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())

LangSmith 추적을 하지 않습니다.


2024-12-23 06:17:40,609 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [3]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:17:53,234 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:17:53,235 - recipe_logger - INFO - langfuse prompt template 생성 완료


2024-12-23 06:17:53,235 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 06:17:54,692 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:17:54,704 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:17:54,705 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:17:54,706 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:18:44,795 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:18:44,809 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:18:44,809 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:18:44,810 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:18:46,404 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [4]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df[col_name+'score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df[col_name+'score'].mean()

# 3. score 하위 10개의 행
top3_scores = df.nsmallest(10, col_name+'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nScore 하위 10개의 행:")
print(top3_scores)

Score 평균값: 0.8699999999999999

Score 하위 10개의 행:
                                            user_info  \
17  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
27  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
33  {'user_allergy_ingredients': [], 'user_dislike...   
4   {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
0   {'user_allergy_ingredients': [], 'user_dislike...   
1   {'user_allergy_ingredients': [], 'user_dislike...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   
6   {'user_allergy_ingredients': [], 'user_dislike...   
7   {'user_allergy_ingredients': ['우유', '크림', '버터'...   

                                          recipe_info  \
17  {'_id': '6761069a846f9e5eb97606ed', 'title': '...   
27  {'_id': '6761069a846f9e5eb97610a5', 'title': '...   
33  {'_id': '6761069a846f9e5eb9761f26', 'title': '...   
4   {'_id': '6761069a846f9e5eb976093b', 'title': '...   
0   {'_id': '67610699846f9e5eb975e532',

In [13]:
df.iloc[10]

user_info                  {'user_allergy_ingredients': [], 'user_dislike...
recipe_info                {'_id': '6761069a846f9e5eb9761fe5', 'title': '...
generation                 {'main_changes_from_original_recipe': ['🥣 닭가슴살...
groundTruth                {'main_changes_from_original_recipe': ['🥦 브로콜리...
contextcorrectness         {'score': 0.2, 'reason': 'The generation does ...
contextcorrectnessscore                                                  0.2
Name: 10, dtype: object

In [5]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:19:51,606 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:19:51,607 - recipe_logger - INFO - langfuse prompt template 생성 완료


2024-12-23 06:19:51,615 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 06:19:52,967 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:19:52,978 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:19:52,979 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:19:52,980 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:20:42,425 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:20:42,437 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:20:42,438 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:42,439 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:20:43,765 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [6]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df[col_name+'score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df[col_name+'score'].mean()

# 3. score 하위 10개의 행
top3_scores = df.nsmallest(10, col_name+'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nScore 하위 10개의 행:")
print(top3_scores)

Score 평균값: 0.8885714285714285

Score 하위 10개의 행:
                                            user_info  \
6   {'user_allergy_ingredients': [], 'user_dislike...   
8   {'user_allergy_ingredients': [], 'user_dislike...   
13  {'user_allergy_ingredients': [], 'user_dislike...   
17  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
27  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
29  {'user_allergy_ingredients': ['새우', '게', '견과류'...   
0   {'user_allergy_ingredients': [], 'user_dislike...   
1   {'user_allergy_ingredients': [], 'user_dislike...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   

                                          recipe_info  \
6   {'_id': '6761069a846f9e5eb975f670', 'title': '...   
8   {'_id': '6761069a846f9e5eb9760f44', 'title': '...   
13  {'_id': '6761069a846f9e5eb97622b1', 'title': '...   
17  {'_id': '6761069a846f9e5eb97606ed', 'title': '...   
27  {'_id': '6761069a846f9e5eb97610a5',

In [7]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:20:43,973 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 06:20:43,975 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:43,982 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 06:20:45,498 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:20:45,512 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:20:45,513 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:45,514 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 06:20:47,135 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:20:47,148 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:20:47,148 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:47,149 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:21:58,000 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:21:58,018 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:21:58,018 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:21:58,020 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:22:00,972 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [9]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df[col_name+'score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df[col_name+'score'].mean()

# 3. score 하위 10개의 행
top3_scores = df.nsmallest(10, col_name+'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nScore 하위 10개의 행:")
print(top3_scores)

Score 평균값: 0.8542857142857143

Score 하위 10개의 행:
                                            user_info  \
5   {'user_allergy_ingredients': [], 'user_dislike...   
17  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
29  {'user_allergy_ingredients': ['새우', '게', '견과류'...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   
8   {'user_allergy_ingredients': [], 'user_dislike...   
9   {'user_allergy_ingredients': ['새우', '게', '견과류'...   
11  {'user_allergy_ingredients': [], 'user_dislike...   
34  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
1   {'user_allergy_ingredients': [], 'user_dislike...   

                                          recipe_info  \
5   {'_id': '67610699846f9e5eb975e745', 'title': '...   
17  {'_id': '6761069a846f9e5eb97606ed', 'title': '...   
29  {'_id': '6761069a846f9e5eb97600e1', 'title': '...   
2   {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3   {'_id': '6761069a846f9e5eb9760639',

### 2. eval prompt로 test - relevance
- 1,2,3번 모두 0.0

In [1]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


LangSmith 추적을 하지 않습니다.


2024-12-23 07:42:37,806 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [2]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 07:42:41,147 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 07:42:41,725 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:42:41,729 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 07:42:45,639 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:42:45,649 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:42:45,650 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:42:45,651 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 07:42:47,027 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:42:47,042 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:42:47,043 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:42:47,044 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 07:45:10,605 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:45:10,622 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:45:10,623 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:45:10,624 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 07:45:19,770 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [5]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8057142857142858
{'user_allergy_ingredients': ['새우', '게', '견과류'], 'user_dislike_ingredients': ['닭고기', '브로콜리', '오이'], 'user_spicy_level': '5단계', 'user_cooking_level': '고급', 'user_owned_ingredients': ['연어', '아스파라거스', '파스타면'], 'user_basic_seasoning': ['버터', '허브솔트', '파마산 치즈'], 'must_use_ingredients': ['아스파라거스']}
{'_id': '6761069a846f9e5eb976030a', 'title': '\ufeff편스토랑이찬원 멸치고추다짐장 레시피 만드는법', 'type_key': '반찬', 'method_key': '조림', 'servings': '4인분', 'cooking_time': '30분 이내', 'difficulty': '초급', 'ingredients': ['멸치(60마리)', '아삭이고추(2개)', '홍고추(2개)', '청양고추(16개)', '식용유(4큰술)', '다진 마늘(4큰술)', '국간장(4큰술)', '멸치액젓(3큰술)', '매실액(1큰술)', '참기름(2큰술)', '물(1/2컵)'], 'cooking_steps': ['멸치를 내장과 머리를 제거하고 다져 줍니다. 그리고 마른 팬에 5분 정도 볶아서 비린내와 수분을 날려 주세요.약 불에서 볶아 주세요.', '고추는 모두 잘게 다져 줍니다.', '다시마로 육수를 내어 줍니다.센 불에서 끓여 주세요.', '달궈 진 팬에 식용유를 두르고 다진 마늘을 넣어서 볶아 줍니다. 그리고 식용유에 마늘 향이 베이면 다진 고추를 넣고 볶아 주세요.중 약불에서 볶아 주세요', '고추가 어느 정도 볶아지면 멸치를 넣은 후 양념 재료와 다시마 육수를 적당히 넣어서 볶아 줍니다.중 약불에서 볶아 주세요.', ' 참기름을 넣고 한번 더 볶은 후 불을 꺼 줍니다.중 약

In [6]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 07:47:53,582 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:47:53,583 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:47:53,587 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 07:47:54,870 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:47:54,882 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:47:54,883 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:47:54,884 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 07:47:58,775 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:47:58,785 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:47:58,786 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:47:58,786 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 14 (Attempt 1)...


2024-12-23 07:48:50,143 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:48:50,158 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:48:50,158 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:48:50,159 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 15 (Attempt 1)...


2024-12-23 07:48:57,570 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:48:57,579 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:48:57,580 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:48:57,580 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 16 (Attempt 1)...


In [14]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.6842857142857143

Reason이 '해당없음'가 아닌 행:
                                            user_info  \
0   {'user_allergy_ingredients': [], 'user_dislike...   
1   {'user_allergy_ingredients': [], 'user_dislike...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   
4   {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
5   {'user_allergy_ingredients': [], 'user_dislike...   
6   {'user_allergy_ingredients': [], 'user_dislike...   
7   {'user_allergy_ingredients': ['우유', '크림', '버터'...   
8   {'user_allergy_ingredients': [], 'user_dislike...   
9   {'user_allergy_ingredients': ['새우', '게', '견과류'...   
10  {'user_allergy_ingredients': [], 'user_dislike...   
11  {'user_allergy_ingredients': [], 'user_dislike...   
12  {'user_allergy_ingredients': [], 'user_dislike...   
13  {'user_allergy_ingredients': [], 'user_dislike...   
14  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
15  {'user_allergy_ingredients': []

In [15]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:44:42,827 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:44:42,828 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:44:42,828 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 06:44:48,879 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:44:48,890 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:44:48,891 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:44:48,891 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 06:44:50,443 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:44:50,459 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:44:50,460 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:44:50,462 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:46:49,788 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:46:49,799 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:46:49,800 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:46:49,801 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:46:54,291 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [16]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.89

Reason이 '해당없음'가 아닌 행:
                                            user_info  \
0   {'user_allergy_ingredients': [], 'user_dislike...   
1   {'user_allergy_ingredients': [], 'user_dislike...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   
4   {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
5   {'user_allergy_ingredients': [], 'user_dislike...   
6   {'user_allergy_ingredients': [], 'user_dislike...   
7   {'user_allergy_ingredients': ['우유', '크림', '버터'...   
8   {'user_allergy_ingredients': [], 'user_dislike...   
9   {'user_allergy_ingredients': ['새우', '게', '견과류'...   
10  {'user_allergy_ingredients': [], 'user_dislike...   
11  {'user_allergy_ingredients': [], 'user_dislike...   
12  {'user_allergy_ingredients': [], 'user_dislike...   
13  {'user_allergy_ingredients': [], 'user_dislike...   
14  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
15  {'user_allergy_ingredients': [], 'user_dislik